In [1]:
import warnings

import skrub
from sklearn.ensemble import HistGradientBoostingClassifier
from skrub import TableVectorizer

import sempipes

warnings.filterwarnings("ignore")


dataset = skrub.datasets.fetch_midwest_survey()


responses = skrub.var("responses")
responses = responses.skb.set_description(dataset.metadata["description"])

labels = skrub.var("labels")
labels = labels.skb.set_description(dataset.metadata["target"])

responses = responses.skb.mark_as_X()
labels = labels.skb.mark_as_y()

with_demographic_features = responses.sem_gen_features(
    nl_prompt="""
        Compute additional demographics-related features, use your intrinsic knowledge about the US. 
        Take into account how the identification with the country or regions of it changed over the generations.         
        Also think about how the identification differs per class and education. The midwest is generally associated 
        with "Midwestern values" — friendliness, modesty, hard work, and community-mindedness.
    """,
    name="demographic_features",
    how_many=5,
)

with_more_demographic_features = with_demographic_features.sem_gen_features(
    nl_prompt="""
        Compute even more demographics-related features!
    """,
    name="more_demographic_features",
    how_many=5,
)

with_spatial_features = responses.sem_gen_features(
    nl_prompt="""
        Compute additional geo-spatial-related features, use your intrinsic knowledge about the US.
    """,
    name="spatial_features",
    how_many=5,
)

all_features = with_more_demographic_features.skb.concat([with_spatial_features], axis=1)

feature_encoder = TableVectorizer()
encoded_responses = all_features.skb.apply(feature_encoder)
pipeline = encoded_responses.skb.apply(HistGradientBoostingClassifier(), y=labels)

In [2]:
pipeline

<Apply HistGradientBoostingClassifier>

In [3]:
from sempipes.optimisers import EvolutionarySearch, MonteCarloTreeSearch, TreeSearch, optimise_colopro

tuning_data = {"responses": dataset.X.head(n=500), "labels": dataset.y.head(n=500)}

outcomes = optimise_colopro(
    pipeline,
    num_trials=16,
    scoring="accuracy",
    search=MonteCarloTreeSearch(),
    cv=2,
    additional_env_variables=tuning_data,
)

2026-02-28 06:56:47,796 - INFO - SEMPIPES> OP-SELECTION> Operators in topological order: ['demographic_features', 'spatial_features', 'more_demographic_features']
2026-02-28 06:56:47,857 - INFO - SEMPIPES> COLOPRO> Computing pipeline summary for context-aware optimisation
2026-02-28 06:56:49,275 - INFO - SEMPIPES> COLOPRO> Processing trial 0
2026-02-28 06:56:49,306 - INFO - SEMPIPES> COLOPRO> Initialising optimisation via OPRO
2026-02-28 06:56:49,327 - INFO - SEMPIPES> COLOPRO> Creating root node
2026-02-28 06:56:49,328 - INFO - SEMPIPES> COLOPRO> Scoring pipeline via 2-fold cross-validation
2026-02-28 06:57:01,924 - INFO - SEMPIPES> COLOPRO> Pipeline scoring took 12.60 seconds
2026-02-28 06:57:01,926 - INFO - SEMPIPES> COLOPRO> Score changed from None to 0.792
2026-02-28 06:57:01,927 - INFO - SEMPIPES> COLOPRO> Processing trial 1
2026-02-28 06:57:01,978 - INFO - SEMPIPES> OP-SELECTION> Selected demographic_features with dependents: ['more_demographic_features']
2026-02-28 06:57:01,979

In [4]:
best_outcome = max(outcomes, key=lambda x: x.score)
best_outcome.score

0.866

In [5]:
from IPython.display import Code, display

display(Code(best_outcome.states.get("demographic_features")["generated_code"]))

import pandas as pd
import numpy as np

def _sem_gen_features(df: pd.DataFrame) -> pd.DataFrame:
    # Create a copy to avoid modifying the original DataFrame and to prevent SettingWithCopyWarning
    df_copy = df.copy()

    # Feature 1: is_self_identified_midwesterner
    # This binary feature indicates whether a respondent explicitly identifies their current living region
    # as the Midwest or a variation thereof. This is a strong direct indicator of their regional identity
    # and is highly relevant for classifying their Census_Region. It leverages real-world knowledge about
    # common ways people refer to the Midwest, including slight variations and typos.
    # Input samples: 'What_would_you_call_the_part_of_the_country_you_live_in_now':
    # ['Southern', 'Midwest', 'Mid-west', 'midewest', 'Mid west', 'The Upper Midwest']
    midwest_keywords = ['midwest', 'mid-west', 'mid west', 'upper midwest', 'midewest']
    df_copy.loc[:, 'is_self_identified_midwesterner'] = df_copy['What_would_you_call_the_part_of_the_country_you_live_in_now'].str.lower().str.contains('|'.join(midwest_keywords), na=False).astype(int)

    # Feature 2: midwest_state_agreement_count
    # This feature counts how many of the commonly recognized Midwestern states (Illinois, Indiana, Iowa, Kansas,
    # Michigan, Minnesota, Missouri, Nebraska, North Dakota, Ohio, South Dakota, Wisconsin) a respondent
    # considers part of the Midwest. A higher count suggests a stronger or more traditional understanding
    # of the Midwest, reflecting deeper regional knowledge or affiliation. This can help differentiate
    # between individuals with varying degrees of "Midwestern" identity, which is crucial for predicting
    # their Census_Region.
    # Input samples: 'Do_you_consider_Illinois_state_as_part_of_the_Midwest': ['No', 'Yes', 'Yes'],
    # 'Do_you_consider_Indiana_state_as_part_of_the_Midwest': ['No', 'No', 'Yes'],
    # 'Do_you_consider_Iowa_state_as_part_of_the_Midwest': ['No', 'Yes', 'No']
    midwest_states_cols = [
        'Do_you_consider_Illinois_state_as_part_of_the_Midwest',
        'Do_you_consider_Indiana_state_as_part_of_the_Midwest',
        'Do_you_consider_Iowa_state_as_part_of_the_Midwest',
        'Do_you_consider_Kansas_state_as_part_of_the_Midwest',
        'Do_you_consider_Michigan_state_as_part_of_the_Midwest',
        'Do_you_consider_Minnesota_state_as_part_of_the_Midwest',
        'Do_you_consider_Missouri_state_as_part_of_the_Midwest',
        'Do_you_consider_Nebraska_state_as_part_of_the_Midwest',
        'Do_you_consider_North_Dakota_state_as_part_of_the_Midwest',
        'Do_you_consider_Ohio_state_as_part_of_the_Midwest',
        'Do_you_consider_South_Dakota_state_as_part_of_the_Midwest',
        'Do_you_consider_Wisconsin_state_as_part_of_the_Midwest'
    ]
    # Convert 'Yes'/'No' to 1/0 for summation
    for col in midwest_states_cols:
        df_copy.loc[:, col + '_numeric'] = df_copy[col].map({'Yes': 1, 'No': 0})
    df_copy.loc[:, 'midwest_state_agreement_count'] = df_copy[[col + '_numeric' for col in midwest_states_cols]].sum(axis=1)
    # Drop the temporary numeric columns
    df_copy = df_copy.drop(columns=[col + '_numeric' for col in midwest_states_cols])

    # Feature 3: personal_midwestern_id_score
    # This ordinal numerical feature quantifies the strength of a respondent's personal identification
    # as a Midwesterner. This direct measure of self-identification is crucial for understanding
    # their regional affinity and is expected to be a strong predictor for Census_Region. It directly
    # translates a subjective feeling into a quantifiable score.
    # Input samples: 'How_much_do_you_personally_identify_as_a_Midwesterner':
    # ['Not much', 'A lot', 'Some', 'Not at all']
    identification_mapping = {
        'Not at all': 0,
        'Not much': 1,
        'Some': 2,
        'A lot': 3
    }
    df_copy.loc[:, 'personal_midwestern_id_score'] = df_copy['How_much_do_you_personally_identify_as_

In [6]:
display(Code(best_outcome.states.get("more_demographic_features")["generated_code"]))

import pandas as pd
import numpy as np

def _sem_gen_features(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()

    # Define all necessary mappings at the beginning of the function
    # (Mapping for age_numeric)
    age_mapping = {
        '18-29': 0,
        '30-44': 1,
        '45-60': 2,
        '60+': 3,
        'Prefer not to answer': np.nan
    }
    # (Mapping for income_numeric)
    income_mapping = {
        '$0 - $24,999': 0,
        '$25,000 - $49,999': 1,
        '$50,000 - $99,999': 2,
        '$100,000 - $149,999': 3,
        '$150,000+': 4,
        'Prefer not to answer': np.nan
    }
    # (Mapping for education_numeric)
    education_mapping = {
        'Less than high school': 0,
        'High school degree': 1,
        'Some college': 2,
        'Associate or bachelor degree': 3,
        'Graduate degree': 4,
        'Prefer not to answer': np.nan
    }
    # (Mapping for gender_encoded)
    gender_mapping = {
        'Male': 0,
        'Female': 1,
        'Prefer not to answer': np.nan
    }

    # Generate base numerical demographic features first
    # (Feature 1: age_numeric)
    # This feature converts the categorical 'Age' column into an ordinal numerical representation.
    # Age is a fundamental demographic factor that often correlates with opinions, life experiences,
    # and regional identity. A numerical representation allows the model to capture this ordering.
    # Input samples: 'Age': ['18-29', '18-29', '18-29', '18-29', '30-44', '30-44', '18-29', '18-29', '18-29', '30-44']
    df_copy['age_numeric'] = df_copy['Age'].map(age_mapping)

    # (Feature 2: income_numeric)
    # This feature converts the categorical 'Household_Income' column into an ordinal numerical representation.
    # Income level is a key socioeconomic indicator that can influence lifestyle, mobility, and regional affiliation.
    # Converting it to a numerical scale provides an ordered feature that can reveal trends.
    # Input samples: 'Household_Income': ['$50,000 - $99,999', '$0 - $24,999', '$25,000 - $49,999', '$50,000 - $99,999', '$150,000+', '$150,000+', '$150,000+', '$100,000 - $149,999', '$100,000 - $149,999', '$50,000 - $99,999']
    df_copy['income_numeric'] = df_copy['Household_Income'].map(income_mapping)

    # (Feature 3: education_numeric)
    # This feature converts the categorical 'Education' column into an ordinal numerical representation.
    # Educational attainment is another crucial demographic factor that can correlate with cultural views,
    # information consumption, and self-identification with regions.
    # Input samples: 'Education': ['High school degree', 'Associate or bachelor degree', 'Associate or bachelor degree', 'Associate or bachelor degree', 'Graduate degree', 'Associate or bachelor degree', 'Some college', 'High school degree', 'Some college', 'Associate or bachelor degree']
    df_copy['education_numeric'] = df_copy['Education'].map(education_mapping)

    # (Feature 4: gender_encoded)
    # This feature converts the categorical 'Gender' column into a numerical representation.
    # Gender is a fundamental demographic attribute that can influence perspectives and regional self-identification.
    # Input samples: 'Gender': ['Male', 'Male', 'Male', 'Male', 'Male', 'Male', 'Male', 'Male', 'Male', 'Male']
    df_copy['gender_encoded'] = df_copy['Gender'].map(gender_mapping)

    # (Feature 5: zip_first_digit)
    # This feature extracts the first digit from the 'In_what_ZIP_code_is_your_home_located' column.
    # The first digit of a US ZIP code broadly indicates a specific geographic region or group of states.
    # This provides a coarse but powerful geographic demographic proxy, highly relevant for predicting "Census_Region".
    # The previous error 'could not convert string to float: 'N'' is handled by coercing non-numeric values to NaN.
    # Input samples: 'In_what_ZIP_code_is_your_home_located': ['74070', '44106', '48185', '45040', '44054', '61422', '63376', '60202'

In [7]:
display(Code(best_outcome.states.get("spatial_features")["generated_code"]))

def _sem_gen_features(df):
    return df